# Homework 2 — Classification

> **Study/reference notebook:** The assignment asks students to write their own code. Use this notebook to study the calculations, then complete the designated cells in the original Coursera notebook. Preserve the original notebook name, cells, metadata, function signatures, and data paths when submitting.

This assignment practices decision-tree attribute selection using entropy and information gain, and Naive Bayes conditional probability estimation. The sample CSV is expected at `./data/dataset.csv`.

## Chapter 2 concepts used here

Classification is supervised learning: the model learns from examples with known class labels. For a decision tree, entropy measures class uncertainty, `H = -Σ p(c) log₂ p(c)`. Information gain for an attribute is the target entropy minus the weighted entropy of the partitions made by that attribute; the attribute with the largest gain is selected.

Naive Bayes estimates class priors `P(Loan)` and conditional probabilities `P(attribute=value | Loan)`, then combines them under the conditional-independence assumption. This assignment asks for the empirical conditional probabilities without Laplace smoothing, consistent with its expected table (which includes zero probabilities).

## Assignment rules and imports

Do not install extra packages. Keep the provided relative dataset path and function APIs. The dataset and hidden grader are supplied by Coursera and are not included in this reference notebook.

In [ ]:
import argparse
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from collections import defaultdict
import math
import operator

## Problem 1 — decision tree (10 points)

The target variable is `Loan`, with classes `yes` and `no`. `information_gain_target()` computes the entropy of the target distribution. The helper `information_gain(p_count_yes, p_count_no)` calculates entropy from class counts. `information_gain_attributes()` computes the weighted conditional entropy for every candidate attribute and returns each information gain plus the highest-gain attribute.

In [ ]:
def information_gain_target(dataset_file):
    df = pd.read_csv(dataset_file)
    loan_counts = df['Loan'].value_counts()
    total = loan_counts.get('yes', 0) + loan_counts.get('no', 0)
    ig_loan = 0.0
    if total > 0:
        for label in ('yes', 'no'):
            count = loan_counts.get(label, 0)
            if count > 0:
                probability = count / total
                ig_loan -= probability * math.log2(probability)
    return ig_loan

In [ ]:
def information_gain(p_count_yes, p_count_no):
    total = p_count_yes + p_count_no
    if total == 0:
        return 0.0

    ig = 0.0
    for count in (p_count_yes, p_count_no):
        if count > 0:
            probability = count / total
            ig -= probability * math.log2(probability)
    return ig

In [ ]:
attribute_values = {
    'Age': ['<=30', '31-40', '>40'],
    'Income': ['low', 'medium', 'high'],
    'Student': ['yes', 'no'],
    'Credit Rating': ['fair', 'excellent']
}
attributes = ['Age', 'Income', 'Student', 'Credit Rating']

def information_gain_attributes(dataset_file, ig_loan, attributes, attribute_values):
    results = {
        'ig_attributes': {attribute: 0 for attribute in attributes},
        'best_attribute': ''
    }
    df = pd.read_csv(dataset_file)
    d_range = len(df)

    for attribute in attributes:
        conditional_entropy = 0.0
        for att_value in attribute_values[attribute]:
            subset = df[df[attribute] == att_value]
            if d_range == 0 or len(subset) == 0:
                continue
            class_counts = subset['Loan'].value_counts()
            class_total = class_counts.get('yes', 0) + class_counts.get('no', 0)
            entropy = 0.0
            if class_total > 0:
                for label in ('yes', 'no'):
                    count = class_counts.get(label, 0)
                    if count > 0:
                        probability = count / class_total
                        entropy -= probability * math.log2(probability)
            conditional_entropy += (len(subset) / d_range) * entropy

        results['ig_attributes'][attribute] = ig_loan - conditional_entropy

    if results['ig_attributes']:
        results['best_attribute'] = max(
            results['ig_attributes'].items(), key=operator.itemgetter(1)
        )[0]
    return results

For the supplied sample, the prompt reports target entropy `0.9798687566511528`; the expected gains are approximately Age `0.24197`, Income `0.01240`, Student `0.19571`, and Credit Rating `0.07182`. `Age` is therefore the best first split. Run the provided visible tests in the Coursera notebook.

## Problem 2 — Naive Bayes classifier (10 points)

The output table contains the class priors for `Loan` and the conditional probabilities for every listed attribute value given each class. For example, `probabilities['Age']['<=30']['yes']` is `P(Age='<=30' | Loan='yes')`. Denominators are the number of rows in the corresponding Loan class. No smoothing is applied in this exercise.

In [ ]:
def naive_bayes(dataset_file, attributes, attribute_values):
    probabilities = {
        'Age': {'<=30': {'yes': 0, 'no': 0}, '31-40': {'yes': 0, 'no': 0}, '>40': {'yes': 0, 'no': 0}},
        'Income': {'low': {'yes': 0, 'no': 0}, 'medium': {'yes': 0, 'no': 0}, 'high': {'yes': 0, 'no': 0}},
        'Student': {'yes': {'yes': 0, 'no': 0}, 'no': {'yes': 0, 'no': 0}},
        'Credit Rating': {'fair': {'yes': 0, 'no': 0}, 'excellent': {'yes': 0, 'no': 0}},
        'Loan': {'yes': 0, 'no': 0}
    }

    df = pd.read_csv(dataset_file)
    d_range = len(df)
    loan_counts = df['Loan'].value_counts()
    vcount_loan_yes = loan_counts.get('yes', 0)
    vcount_loan_no = loan_counts.get('no', 0)

    if d_range > 0:
        probabilities['Loan']['yes'] = vcount_loan_yes / d_range
        probabilities['Loan']['no'] = vcount_loan_no / d_range

    for attribute in attributes:
        for att_value in attribute_values[attribute]:
            matching = df[df[attribute] == att_value]
            yes_count = (matching['Loan'] == 'yes').sum()
            no_count = (matching['Loan'] == 'no').sum()
            probabilities[attribute][att_value]['yes'] = (
                yes_count / vcount_loan_yes if vcount_loan_yes else 0
            )
            probabilities[attribute][att_value]['no'] = (
                no_count / vcount_loan_no if vcount_loan_no else 0
            )

    return probabilities

## Validation and submission checklist

- Confirm `./data/dataset.csv` has columns `Age`, `Income`, `Student`, `Credit Rating`, and `Loan`.
- Run all visible tests supplied with the original Coursera notebook. The hidden grader and sample dataset are not available in this reference file.
- Keep function signatures unchanged and run code cells in order.
- Submit through the original notebook opened from the assignment page; copy only code into its designated answer cells.